In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, random_split
import numpy as np

### 1. Transformations Setup

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

### 2. Dataset Loading & Stratified Random Subset

In [ ]:
dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\cv_data\data"
full_dataset = ImageFolder(root=dataset_path, transform=transform)

### Random indices so all 10 gesture classes are included

In [ ]:
total_samples = len(full_dataset)
np.random.seed(42)
sample_size = 30000
random_indices = np.random.choice(total_samples, size=sample_size, replace=False)

dataset = Subset(full_dataset, indices=random_indices)

### Train/Validation Split (80% Train, 20% Val)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

### DataLoaders

In [ ]:
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=0)

### 3. Model Architecture

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(16 * 16 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 3)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

### Build Model

In [ ]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 4. Fixed Training Loop

In [ ]:
epochs = 9
model.train()

print("--- Training Started ---")
for epoch in range(epochs):
    epoch_train_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()      
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()
    
    avg_loss = epoch_train_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}")


# 5. Validation Evaluation

In [ ]:
model.eval() 
correct_labels = 0
total_labels = 0

with torch.no_grad(): 
    for images, labels in val_loader:
        outputs = model(images)                
        _, predicted = torch.max(outputs, 1)   
        correct_labels += (predicted == labels).sum().item()  
        total_labels += labels.size(0)                        

accuracy = (correct_labels / total_labels) * 100
print(f"\nValidation Accuracy: {accuracy:.2f}%")

# Save Model Weights
torch.save(model.state_dict(), 'hand_gesture_cnn.pth')
print("Model saved successfully as 'hand_gesture_cnn.pth'!")